In [1]:
import pandas as pd
from os.path import join, abspath
from bs4 import BeautifulSoup
from bs4.element import NavigableString
import numpy as np
from IPython.display import display
import re
from requests import get
from bs4 import BeautifulSoup
from urllib.parse import urlparse
from copy import deepcopy

artifacts_abspath = abspath("../../../artifacts")
assets_abspath = abspath("../../../assets")
source_abspath = join(assets_abspath, "evm/opcodes.xlsx")
target_abspath = join(artifacts_abspath, "evm/opcodes_with_unique_card.csv")
options = [
    "display.max_rows",
    None,
    "display.max_columns",
    None,
    "display.max_colwidth",
    None,
]

In [2]:
notes_remove = "(opens in a new tab)"
source = pd.read_excel(source_abspath, sheet_name="Sheet")
source.rename(
    columns={
        "Stack": "Opcode",
        "Name": "Mnemonic",
        "Initial Stack": "Inputs",
        "Resulting Stack": "Output",
    },
    inplace=True,
)


def notes_rewrite(v):
    if pd.isna(v):
        return v
    v = v.replace(notes_remove, "")
    v = v[0].upper() + v[1:]
    v = v.replace("Uint", "uint")
    return v


source["Notes"] = source["Notes"].apply(notes_rewrite)
with pd.option_context(*options):
    display(source)

,Opcode,Mnemonic,Gas Constant,Gas Dynamic,Inputs,Output,Mem / Storage,Notes
0,00,STOP,0,NaN,NaN,NaN,NaN,Halt execution
1,01,ADD,3,NaN,"a, b",a + b,NaN,(u)int256 addition modulo 2**256
2,02,MUL,5,NaN,"a, b",a * b,NaN,(u)int256 multiplication modulo 2**256
3,03,SUB,3,NaN,"a, b",a - b,NaN,(u)int256 addition modulo 2**256
4,04,DIV,5,NaN,"a, b",a // b,NaN,uint256 division
5,05,SDIV,5,NaN,"a, b",a // b,NaN,Int256 division
6,06,MOD,5,NaN,"a, b",a % b,NaN,uint256 modulus
7,07,SMOD,5,NaN,"a, b",a % b,NaN,Int256 modulus
8,08,ADDMOD,8,NaN,"a, b, N",(a + b) % N,NaN,(u)int256 addition modulo N
9,09,MULMOD,8,NaN,"a, b, N",(a * b) % N,NaN,(u)int256 multiplication modulo N


In [3]:
# Merge Push entries
push = source.copy()
pushes = push[push["Mnemonic"].apply(lambda v: "PUSH" in v and v != "PUSH0")]
first = pushes.iloc[0]
last = pushes.iloc[-1]
merged = first.copy()
merged.update(
    pd.Series(
        {
            "Opcode": str(first["Opcode"]) + ".." + str(last["Opcode"]),
            "Mnemonic": "PUSH[n]",
            "Output": "uint[8*n]",
            "Notes": "push n-byte value onto stack, where n is between 1..32 inclusive",
        }
    )
)
push.drop(index=list(pushes.index), inplace=True)
push.loc[len(push.index)] = merged
push.reset_index(inplace=True, drop=True)
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(push)

,Opcode,Mnemonic,Gas Constant,Gas Dynamic,Inputs,Output,Mem / Storage,Notes
0,00,STOP,0,NaN,NaN,NaN,NaN,Halt execution
1,01,ADD,3,NaN,"a, b",a + b,NaN,(u)int256 addition modulo 2**256
2,02,MUL,5,NaN,"a, b",a * b,NaN,(u)int256 multiplication modulo 2**256
3,03,SUB,3,NaN,"a, b",a - b,NaN,(u)int256 addition modulo 2**256
4,04,DIV,5,NaN,"a, b",a // b,NaN,uint256 division
5,05,SDIV,5,NaN,"a, b",a // b,NaN,Int256 division
6,06,MOD,5,NaN,"a, b",a % b,NaN,uint256 modulus
7,07,SMOD,5,NaN,"a, b",a % b,NaN,Int256 modulus
8,08,ADDMOD,8,NaN,"a, b, N",(a + b) % N,NaN,(u)int256 addition modulo N
9,09,MULMOD,8,NaN,"a, b, N",(a * b) % N,NaN,(u)int256 multiplication modulo N


In [4]:
swap = push.copy()
swaps = swap[swap["Mnemonic"].str.contains("SWAP", case=False)].copy()
first = swaps.iloc[0]
last = swaps.iloc[-1]
swaps["Inputs"] = last["Inputs"]
swaps["Output"] = last["Output"]
merged = last.copy()
merged.update(
    pd.Series(
        {
            "Opcode": str(first["Opcode"]) + ".." + str(last["Opcode"]),
            "Mnemonic": "SWAP[n]",
            "Notes": "swap top of the stack with the nth item",
        }
    )
)
swap.drop(index=swaps.index, inplace=True)
swap.reset_index(inplace=True, drop=True)
swap.loc[len(swap.index)] = merged
display(swap)

,Opcode,Mnemonic,Gas Constant,Gas Dynamic,Inputs,Output,Mem / Storage,Notes
0,00,STOP,0,NaN,NaN,NaN,NaN,Halt execution
1,01,ADD,3,NaN,"a, b",a + b,NaN,(u)int256 addition modulo 2**256
2,02,MUL,5,NaN,"a, b",a * b,NaN,(u)int256 multiplication modulo 2**256
3,03,SUB,3,NaN,"a, b",a - b,NaN,(u)int256 addition modulo 2**256
4,04,DIV,5,NaN,"a, b",a // b,NaN,uint256 division
...,...,...,...,...,...,...,...,...
105,FB-FC,invalid,NaN,NaN,NaN,NaN,NaN,NaN
106,FD,REVERT,NaN,https://github.com/wolflo/evm-opcodes/blob/mai...,"ost, len",.,NaN,Revert(mem[ost:ost+len-1])
107,FE,INVALID,NaN,https://github.com/wolflo/evm-opcodes/blob/mai...,NaN,NaN,NaN,Designated invalid opcode - EIP-141
108,FF,SELFDESTRUCT,NaN,https://github.com/wolflo/evm-opcodes/blob/mai...,addr,.,NaN,Sends all ETH to addr; if executed in the same...


In [5]:
dup = swap.copy()
dups = dup[dup["Mnemonic"].str.contains("DUP")].copy()
first = dups.iloc[0]
last = dups.iloc[-1]
merged = last.copy()
merged.update(
    pd.Series(
        {
            "Mnemonic": "DUP[n]",
            "Opcode": str(first["Opcode"]) + ".." + str(last["Opcode"]),
            "Inputs": last["Inputs"],
            "Output": last["Output"],
            "Notes": "clone nth value to the top of the stack",
        }
    )
)
dup.drop(index=dups.index, inplace=True)
dup.reset_index(inplace=True, drop=True)
dup.loc[len(dup.index)] = merged

with pd.option_context(*options):
    display(dup[dup["Mnemonic"] == "SLOAD"])

,Opcode,Mnemonic,Gas Constant,Gas Dynamic,Inputs,Output,Mem / Storage,Notes
62,54,SLOAD,NaN,https://github.com/wolflo/evm-opcodes/blob/main/gas.md#a6-sload,key,storage[key],NaN,Read word from storage


In [6]:
# full_url = dup.iloc[-3]["Gas Dynamic"]
# gas_url = re.sub(r"#.*", "", full_url)
# response = get(gas_url)
# if response.status_code != 200:
#     raise RuntimeError("Failed fetch")
with open("../../../assets/evm/gas-costs.html", "r") as f:
    response = f.read()
page = BeautifulSoup(response, "html.parser")
page

<article class="markdown-body entry-content container-lg" itemprop="text">
<div class="markdown-heading" dir="auto">
<h1 class="heading-element" dir="auto" tabindex="-1">Appendix - Dynamic Gas Costs</h1><a aria-label="Permalink: Appendix - Dynamic Gas Costs" class="anchor" href="#appendix---dynamic-gas-costs" id="user-content-appendix---dynamic-gas-costs"><svg aria-hidden="true" class="octicon octicon-link" height="16" version="1.1" viewbox="0 0 16 16" width="16">
<path d="m7.775 3.275 1.25-1.25a3.5 3.5 0 1 1 4.95 4.95l-2.5 2.5a3.5 3.5 0 0 1-4.95 0 .751.751 0 0 1 .018-1.042.751.751 0 0 1 1.042-.018 1.998 1.998 0 0 0 2.83 0l2.5-2.5a2.002 2.002 0 0 0-2.83-2.83l-1.25 1.25a.751.751 0 0 1-1.042-.018.751.751 0 0 1-.018-1.042Zm-4.69 9.64a1.998 1.998 0 0 0 2.83 0l1.25-1.25a.751.751 0 0 1 1.042.018.751.751 0 0 1 .018 1.042l-1.25 1.25a3.5 3.5 0 1 1-4.95-4.95l2.5-2.5a3.5 3.5 0 0 1 4.95 0 .751.751 0 0 1-.018 1.042.751.751 0 0 1-1.042.018 1.998 1.998 0 0 0-2.83 0l-2.5 2.5a1.998 1.998 0 0 0 0 2.83Z"

In [7]:
def new_section():
    return BeautifulSoup("", "html.parser")


def append_to_sections(header, section):
    href = header.a["href"][1:]
    section_sanitized = section.svg.decompose()
    section.a["href"] = (
        "https://github.com/wolflo/evm-opcodes/blob/main/gas.md"
    ) + section.a["href"]
    sections[href] = {
        "header": header,
        "section": section,
        "href": href,
        "sections_pretty": section.prettify(),
    }


article = deepcopy(page.article)
sections = {}
section = new_section()
header = article.find(class_="markdown-heading")

for child in article.children:
    if child.name is None:
        continue
    if child.get("class") == ["markdown-heading"]:
        if child.find("h3") is not None or child.find("h2") is not None:
            append_to_sections(header, section)
            section = new_section()
            section.append(child)
            header = child
            continue
    section.append(child)
append_to_sections(header, section)

for key in sections.keys():
    print(key)
    print(sections[key]["section"])
    print("\n\n")

appendix---dynamic-gas-costs
<div class="markdown-heading" dir="auto">
<h1 class="heading-element" dir="auto" tabindex="-1">Appendix - Dynamic Gas Costs</h1><a aria-label="Permalink: Appendix - Dynamic Gas Costs" class="anchor" href="https://github.com/wolflo/evm-opcodes/blob/main/gas.md#appendix---dynamic-gas-costs" id="user-content-appendix---dynamic-gas-costs"></a>
</div>



a0-0-intrinsic-gas
<div class="markdown-heading" dir="auto">
<h3 class="heading-element" dir="auto" tabindex="-1">A0-0: Intrinsic Gas</h3><a aria-label="Permalink: A0-0: Intrinsic Gas" class="anchor" href="https://github.com/wolflo/evm-opcodes/blob/main/gas.md#a0-0-intrinsic-gas" id="user-content-a0-0-intrinsic-gas"></a>
</div><p dir="auto">Intrinsic gas is the amount of gas paid prior to execution of a transaction.
    That is, the gas paid by the initiator of a transaction, which will always be an externally-owned account, before
    any state updates are made or any code is executed.</p><p dir="auto">Gas Calc

In [8]:
gas = dup.copy()


def combine_gas(row):
    constant = row["Gas Constant"]
    dynamic = row["Gas Dynamic"]
    text = ""
    if not pd.isna(constant):
        text += str(constant)
    if not pd.isna(dynamic):
        if not pd.isna(constant):
            text += " + "
        text += "dynamic"
    return text


def fetch_dynamic(row):
    url = str(row["Gas Dynamic"])
    if url == "nan":
        return "<p>This opcode is not dynamic</p>"
    fragment = urlparse(url).fragment
    section = sections[fragment]["sections_pretty"]
    return section


gas.insert(
    gas.columns.get_loc("Mnemonic") + 1, "Gas", gas.apply(combine_gas, axis=1)
)

gas["Dynamic Gas Notes"] = gas.apply(fetch_dynamic, axis=1)
gas.insert(
    len(gas.columns),
    "Dynamic Gas Reference",
    gas["Gas Dynamic"].apply(
        lambda v: pd.NA if pd.isna(v) else f'<a href="{v}">{v}</a>'
    ),
)
gas.drop(columns=["Gas Constant", "Gas Dynamic"], inplace=True)
# gas.rename(columns={
#     "Gas Dynamic": "Gas Reference Url"
# }, inplace=True)


# gas
# with pd.option_context(*options):
#     display(gas)
display(gas)

,Opcode,Mnemonic,Gas,Inputs,Output,Mem / Storage,Notes,Dynamic Gas Notes,Dynamic Gas Reference
0,00,STOP,0,NaN,NaN,NaN,Halt execution,<p>This opcode is not dynamic</p>,NaN
1,01,ADD,3,"a, b",a + b,NaN,(u)int256 addition modulo 2**256,<p>This opcode is not dynamic</p>,NaN
2,02,MUL,5,"a, b",a * b,NaN,(u)int256 multiplication modulo 2**256,<p>This opcode is not dynamic</p>,NaN
3,03,SUB,3,"a, b",a - b,NaN,(u)int256 addition modulo 2**256,<p>This opcode is not dynamic</p>,NaN
4,04,DIV,5,"a, b",a // b,NaN,uint256 division,<p>This opcode is not dynamic</p>,NaN
...,...,...,...,...,...,...,...,...,...
90,FD,REVERT,dynamic,"ost, len",.,NaN,Revert(mem[ost:ost+len-1]),"<div class=""markdown-heading"" dir=""auto"">\n <h...","<a href=""https://github.com/wolflo/evm-opcodes..."
91,FE,INVALID,dynamic,NaN,NaN,NaN,Designated invalid opcode - EIP-141,"<div class=""markdown-heading"" dir=""auto"">\n <h...","<a href=""https://github.com/wolflo/evm-opcodes..."
92,FF,SELFDESTRUCT,dynamic,addr,.,NaN,Sends all ETH to addr; if executed in the same...,"<div class=""markdown-heading"" dir=""auto"">\n <h...","<a href=""https://github.com/wolflo/evm-opcodes..."
93,90..9F,SWAP[n],3,"a, ..., b","b, ..., a",NaN,swap top of the stack with the nth item,<p>This opcode is not dynamic</p>,<NA>


In [9]:
pretty = gas.copy()
for col in [
    "Inputs",
    "Output",
    "Mem / Storage",
    "Notes",
    "Dynamic Gas Reference",
]:
    pretty[col] = pretty[col].apply(lambda v: "None" if pd.isna(v) else v)
with pd.option_context(*options):
    display(pretty.head())

,Opcode,Mnemonic,Gas,Inputs,Output,Mem / Storage,Notes,Dynamic Gas Notes,Dynamic Gas Reference
0,00,STOP,0,None,None,None,Halt execution,<p>This opcode is not dynamic</p>,None
1,01,ADD,3,"a, b",a + b,None,(u)int256 addition modulo 2**256,<p>This opcode is not dynamic</p>,None
2,02,MUL,5,"a, b",a * b,None,(u)int256 multiplication modulo 2**256,<p>This opcode is not dynamic</p>,None
3,03,SUB,3,"a, b",a - b,None,(u)int256 addition modulo 2**256,<p>This opcode is not dynamic</p>,None
4,04,DIV,5,"a, b",a // b,None,uint256 division,<p>This opcode is not dynamic</p>,None


In [10]:
with pd.option_context(*options):
    display(print(pretty[pretty["Mnemonic"] == "SLOAD"]["Dynamic Gas Notes"]))

62    <div class="markdown-heading" dir="auto">\n <h2 class="heading-element" dir="auto" tabindex="-1">\n  A6: SLOAD\n </h2>\n <a aria-label="Permalink: A6: SLOAD" class="anchor" href="https://github.com/wolflo/evm-opcodes/blob/main/gas.md#a6-sload" id="user-content-a6-sload">\n </a>\n</div>\n<p dir="auto">\n See\n <a href="#a0-2-access-sets">\n  A0-2\n </a>\n for details on EIP-2929 and\n <code>\n  touched_storage_slots\n </code>\n .\n</p>\n<p dir="auto">\n Terms:\n</p>\n<ul dir="auto">\n <li>\n  <code>\n   context_addr\n  </code>\n  : the address of the current execution context (i.e. what\n  <code>\n   ADDRESS\n  </code>\n  would\n      put on the stack)\n </li>\n <li>\n  <code>\n   target_storage_key\n  </code>\n  : The 32-byte storage index to load from (\n  <code>\n   key\n  </code>\n  in the stack\n      representation)\n </li>\n</ul>\n<p dir="auto">\n Gas Calculation:\n</p>\n<ul dir="auto">\n <li>\n  <code>\n   gas_cost = 100\n  </code>\n  <strong>\n   if\n  </strong>\n  <code>

None

In [12]:
pretty.to_csv(target_abspath, sep="|", index=False, header=False)